In [0]:
path_rental='development_042_silver_sandbox.ricky_dev.weekly_install_base'
path_final='development_042_silver_sandbox.demand_forecast.lease_units_bronz'
like_games_path='development_042_silver_sandbox.demand_forecast.like_games_ai_attr'

In [0]:
df_raw=spark.table(path_rental)
like_games_df=spark.table(like_games_path)
display(df_raw)

In [0]:
import re

def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

original_columns = df_raw.columns
cleaned_columns = [clean_column_name(c) for c in original_columns]

if len(cleaned_columns) != len(set(cleaned_columns)):
    raise ValueError(
        "Column cleaning created duplicate names. Review these mappings: "
        f"{list(zip(original_columns, cleaned_columns))}"
    )

df_clean = df_raw.toDF(*cleaned_columns)

display(spark.createDataFrame(
    [(old, new) for old, new in zip(original_columns, cleaned_columns)],
    ["original_column", "cleaned_column"],
))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampType

CATEGORY_COL = "land_based_gaming_category"
DOMESTIC_COL = "domestic_international"
THEME_COL = "theme_description"

for col_name in [CATEGORY_COL, DOMESTIC_COL, THEME_COL]:
    if col_name not in df_clean.columns:
        raise ValueError(
            f"Missing expected column {col_name!r}. "
            f"Available columns: {df_clean.columns}"
        )

# Prefer common weekly date column names. If none match, look for a single
# date/week-like column and ask for an explicit choice if there are several.
date_candidates = [
    "week_ending_date",
    "week_end_date",
    "week_date",
    "week_ending",
    "week_start_date",
    "date",
    "gaming_date",
    "week",
    "month_date",
    "fiscal_week",
]

date_col = next(
    (c for c in date_candidates if c in df_clean.columns),
    None,
)

if date_col is None:
    possible_date_cols = [
        c for c in df_clean.columns
        if "date" in c or "week" in c or "month" in c
    ]

    if len(possible_date_cols) == 1:
        date_col = possible_date_cols[0]
    else:
        raise ValueError(
            "Could not identify one date column. Set date_col explicitly. "
            f"Possible date columns: {possible_date_cols}"
        )

date_as_string = F.col(date_col).cast("string")

# Handles common date, timestamp, US date, and compact YYYYMMDD formats.
parsed_date = F.coalesce(
    F.to_date(date_as_string),
    F.to_date(date_as_string, "M/d/yyyy"),
    F.to_date(date_as_string, "MM/dd/yyyy"),
    F.to_date(date_as_string, "yyyyMMdd"),
)

filtered_df = (
    df_clean
    .where(
        F.upper(F.col(CATEGORY_COL)).contains("CIII")
        & (F.upper(F.trim(F.col(DOMESTIC_COL))) == "D")
    )
    .withColumn("_source_date", parsed_date)
    .where(F.col("_source_date").isNotNull())
    .withColumn("month", F.trunc(F.col("_source_date"), "month"))
)

monthly_theme_counts = (
    filtered_df
    .groupBy("month", THEME_COL)
    .agg(F.count(F.lit(1)).alias("row_count"))
    .orderBy("month", F.desc("row_count"), THEME_COL)
)

display(monthly_theme_counts)

In [0]:
theme_description_review = (
    filtered_df
    .groupBy(THEME_COL)
    .agg(
        F.count(F.lit(1)).alias("total_row_count"),
        F.countDistinct("month").alias("months_present"),
    )
    .orderBy(F.desc("total_row_count"), THEME_COL)
)

display(theme_description_review)

In [0]:
"""Theme cleaning, exact matches, and partial letter-based review suggestions.

Python standard library for matching; PySpark only for the Databricks wrapper.
"""
import re
import unicodedata
from collections import Counter
from difflib import SequenceMatcher

# Explicit title expansions. Whole tokens only; never replace inside normal words.
TOKEN_REPLACEMENTS = {
    "wof": "wheel of fortune", "wog": "wheel of gold", "mb": "mega bucks", "pros": "prosperity", "lnk": "link",
    "tpl": "triple", "ml": "mystery of the lamp", "gldspn": "golden spin",
    "tgr": "tiger", "drgn": "dragon", "csh": "cash", "rls": "reels",
    "diaspins": "diamond spins", "diaspns": "diamond spins", "diaspn": "diamond spins",
    "frtn": "fortune", "gng": "gong", "dlx": "deluxe", "phnix": "phoenix",
    "dbl": "double", "dia": "diamond", "diam": "diamond", "diams": "diamonds",
    "mgc": "magic", "trs": "treasures", "tresrs": "treasures",
    "rsng": "rising", "rckts": "rockets",
    "etrnllnk": "eternal link", "swml": "super wheelmania",
    "rwb": "red white blue", "rh": "red hot", "rdht": "red hot",
    "rht": "red hot tamales", "rhtamales": "red hot tamales",
    "szl": "sizzling", "szzlng": "sizzling", "sizzlng": "sizzling",
    "pnk": "pink", "pnthr": "panther", "lky": "lucky", "gld": "gold",
    "gldn": "golden", "rchs": "riches",
    "jckpt": "jackpot", "jkpt": "jackpot", "jkpts": "jackpots", "jwl": "jewels",
    "jckpts": "jackpots", "rspn": "respin", "respn": "respin",
    "pkr": "poker", "rlete": "roulette", "rlette": "roulette",
    "hrou": "hot roulette", "hrgd": "hot roll golden dice",
    "ltg": "lucky triple gold", "tfl": "triple fortune link",
    "hpr": "hyper", "hts": "hits", "wld": "wild", "clver": "clover",
    "longshp": "longship", "lghts": "lights", "lgts": "lights",
    "dlg": "dragon lights gold", "fstl": "festival link",
    "hrki": "haruki", "kmko": "kimiko", "knji": "kenji", "skra": "sakura",
    "spn": "spin", "cac": "connect collect", "fg": "free games",
    "blk": "black", "wdw": "widow", "sprstr": "superstar",
    "erptn": "eruption", "fortne": "fortune",
    "lobstrmania": "lobstermania", "lbstermania": "lobstermania",
    "lbstrmnia": "lobstermania", "lobsterm": "lobstermania",
    "lbstrmniaclssichts": "lobstermania classic hits", "lrys": "larrys",
    "mystry": "mystery", "cngo": "congo",
    "enchntd": "enchanted", "uncrn": "unicorn", "rtn": "returns", "ret": "returns",
    "fvrspin": "fever spins", "samuraisteam": "samurai steam",
    "feverspins": "fever spins", "rodeorails": "rodeo rails",
    "sizzling7spiggyparade": "sizzling 7s piggy parade",
    "mny": "money", "mchn": "machine", "amzng": "amazing", "sphr": "sapphire",
    "dncng": "dancing", "phnx": "phoenix", "srng": "soaring",
    "llucky": "lucky", "surefire": "sure fire", "bg": "big", "clsic": "classic",
    "clssic": "classic", "megatwr": "megatower", "hot7s": "hot 7s",
    "tplredhot7s": "triple red hot 7s", "redhot7": "red hot 7s",
    "redhot7s": "red hot 7s", "rh7s": "red hot 7s", "rh7": "red hot 7s",
    "tpldbl": "triple double", "tpldia": "triple diamond", "dbldia": "double diamond",
    "dblgold": "double gold", "tpldbldia": "triple double diamond",
    "tpldblredhotbells": "triple double red hot bells",
    "tpldblstrs": "triple double stars",
    "tpldblredhot7s": "triple double red hot 7s", "tpldblredhot7": "triple double red hot 7s",
    "tplbigwhl6x4x3x2xvip": "triple big wheel 6x4x3x2x vip",
    "pinballgrand": "pinball grand", "chilimania": "chili mania",
    "cashonreels": "cash on reels", "reelboost": "reel boost",
    "cashclmb": "cash climb", "heavenlytreasures": "heavenly treasures",
    "dragonsphere": "dragon sphere", "tigerdragon": "tiger dragon",
    "redhot": "red hot", "stnkn": "stinkin", "rch": "rich", "gn": "gone",
    "hthit": "hot hit", "mn": "moon", "dsrt": "desert", "ngts": "nights",
    "chlmn": "chili mania", "evltn": "evolution", "fghm": "fu gui hao men",
    "hrt": "heart", "mys": "mystical", "myst": "mystery", "bbble": "bubble",
    "bst": "boost", "rdhtrespndlx": "red hot respin deluxe", "prog": "progressive",
    "yng": "ying", "shn": "shen", "fortnes": "fortunes", "fulflld": "fulfilled",
    "btb": "bring the boom", "wlds": "wilds", "clsc": "classic", "ed": "edition",
    "wst": "west", "zcz": "zhao cai zhu", "gttn": "gettin", "pggy": "piggy", "wth": "with",
    "nefturs": "nefturis", "lkylarrys": "lucky larrys", "lobstrmnia": "lobstermania",
    "4x3x2xjackpotplus": "4x3x2x jackpot plus",
    "tpldblrhbls": "triple double red hot bells",
    "stp": "super times pay", "tdd": "triple double diamond",
}

# Deployment / reporting tokens seen in the supplied premium descriptions.
# Keep family names, LINK, and title modifiers such as GRAND/FG.
NOISE_TOKENS = {
    "avp", "ac", "nj", "wap", "qtr", "asc", "acs", "cc", "ccc",
    "curve", "crv", "ccurve", "dnu", "mlp", "drs", "dom", "nyl",
    "po", "rf", "bk", "bc", "cmb", "tm", "bdl", "vid", "video", "mech", "s3k",
}
ATTACHED_CODES = re.compile(
    r"(?:mlp|asc)(?:(?:asc|cc|c|dnu|drs|nyl|crv|vid|po))*\b"
)
CONFIG = re.compile(r"\d+\s*r(?=\d|\s|mg|ml|mc|asc|$)(?:\s*(?:mg|ml|mc))*(?:\s*\d+\s*[lc])*", re.I)
TOKEN_PATTERN = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in sorted(TOKEN_REPLACEMENTS, key=len, reverse=True)) + r")\b"
)

def clean_theme_name(value):
    if value is None:
        return ""
    s = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower().strip()
    if s in {"", "none", "null", "nan", "n/a", "na", "undefined"}:
        return ""
    # Process attached reporting codes before abbreviations (3RML is not a lamp title).
    s = re.sub(r"\bas\s+ccc\b", " ", s)
    s = CONFIG.sub(" ", s)
    s = ATTACHED_CODES.sub(" ", s)
    s = re.sub(r"\bcd\+", " ", s)
    s = re.sub(r"\b\d+\s*(?:ctc|l|c|stn)\b", " ", s)
    s = re.sub(r"\bc\s*&\s*c\b", "connect collect", s)
    s = re.sub(r"(?<=[a-z])\$(?=\W|$)", "s", s)
    s = re.sub(r"['\u2019`]+", "", s).replace("&", " and ")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = " ".join(t for t in s.split() if t not in NOISE_TOKENS)
    s = TOKEN_PATTERN.sub(lambda m: TOKEN_REPLACEMENTS[m.group()], s)
    # Context-specific expansions to avoid applying ambiguous abbreviations globally.
    s = re.sub(r"\bying ci shen\b", "ying cai shen", s)
    s = re.sub(r"\beternal link (warriors|princess) emp\b", r"eternal link \1 empire", s)
    s = re.sub(r"\bmd (?=(?:blue|red) mountain\b)", "miners dream ", s)
    s = re.sub(r"\bpl (?=(?:drakes|marys) treasure\b)", "pirate link ", s)
    s = re.sub(r"\bstinkin rich sl\b", "stinkin rich stink link", s)
    s = re.sub(r"\bsamurai 888 rs\b", "samurai rise 888", s)
    s = re.sub(r"\bht roulette\b", "hot roulette", s)
    s = re.sub(r"\bll (?=lobstermania\b)", "lucky larrys ", s)
    s = re.sub(r"\bgs\b", "golden spin", s) if s.startswith("wheel of fortune ") else s
    s = re.sub(r"\b(red hot|sizzling)\s*7\b", r"\1 7s", s)
    return " ".join(s.split())

# LINK, CLASSIC HITS, family names, multipliers and sequel numbers are retained.
# CLASSIC HITS has no generic-word exclusion or special weight.
def match_tokens(cleaned):
    words = {
        "diamonds": "diamond", "sevens": "7s", "ii": "2", "iii": "3", "iv": "4",
        "larrys": "larry", "games": "game", "hits": "hit", "treasures": "treasure",
        "jackpots": "jackpot", "returns": "return", "clovers": "clover", "bucks": "buck",
    }
    # Explicit spacing equivalents. These do not remove any title words.
    cleaned = re.sub(r"\bda vinci\b", "davinci", cleaned)
    cleaned = re.sub(r"\bbig horn\b", "bighorn", cleaned)
    cleaned = re.sub(r"\bmegabucks\b", "mega bucks", cleaned)
    cleaned = re.sub(r"\blinkcats\b", "link cats", cleaned)
    return [words.get(w, w) for w in cleaned.split() if w not in {"the", "of", "and"}]


def match_key(cleaned):
    # Sorted words support TRIPLE DIAMOND == DIAMOND TRIPLE.
    # Repeated words are retained (DOUBLE DOUBLE is not DOUBLE).
    return "|".join(sorted(match_tokens(cleaned)))


def token_key(cleaned):
    return " ".join(sorted(match_tokens(cleaned)))


def _letter_ratio(a, b, minimum=0.0):
    """Symmetric character similarity, 0 to 1, with safe upper-bound pruning."""
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    forward = SequenceMatcher(None, a, b, autojunk=False)
    if forward.real_quick_ratio() < minimum or forward.quick_ratio() < minimum:
        return 0.0
    reverse = SequenceMatcher(None, b, a, autojunk=False)
    return (forward.ratio() + reverse.ratio()) / 2.0


def _title_letter_score(a, b, minimum=0.0):
    # Sorting accommodates reordered words. A second comparison accommodates
    # joined/split spellings. Neither comparison discards a title word.
    return max(
        _letter_ratio(" ".join(sorted(a)), " ".join(sorted(b)), minimum),
        _letter_ratio("".join(a), "".join(b), minimum),
    )


def _unshared_words(target, candidate):
    shared = Counter(target) & Counter(candidate)

    def subtract_shared(words):
        remaining_shared = shared.copy()
        remaining = []
        for word in words:
            if remaining_shared[word]:
                remaining_shared[word] -= 1
            else:
                remaining.append(word)
        return remaining

    return subtract_shared(target), subtract_shared(candidate)


def compare_title_tokens(target, candidate, min_score=0.80, differing_min_score=0.70):
    """Return (full-title score, method, differing-text score), or None.

    Exact normalized words can appear in any order. Partial suggestions allow
    multiple spelling differences and extra/missing words, provided the full
    title's character similarity reaches min_score. This is a string heuristic,
    not a probability that two records represent the same game.

    When BOTH titles have nonshared words, their letters must also be similar.
    This prevents a shared phrase (any phrase, including CLASSIC HITS) from
    hiding a completely different remaining title. Subset titles have no such
    two-sided difference, but their missing/extra words lower the full score.
    LINK and numbers stay in both comparisons; differences are exposed for review.
    """
    if not target or not candidate:
        return None
    if Counter(target) == Counter(candidate):
        return (1.0, "normalized_exact", None)
    score = _title_letter_score(target, candidate, min_score)
    if score < min_score:
        return None
    left, right = _unshared_words(target, candidate)
    differing_score = None
    if left and right:
        differing_score = _title_letter_score(left, right, differing_min_score)
        if differing_score < differing_min_score:
            return None
    return (score, "partial_fuzzy_review", differing_score)


def build_theme_mapping(theme_rows, game_rows, overrides=None, top_n=5,
                        review_min_score=0.80, differing_min_score=0.70):
    """Match every original join_game_name independently, allowing shared themes.

    Raw descriptions with the same sorted normalized words form one candidate.
    The full ORIGINAL game title is scored; sales aliases cannot bypass it.
    Fuzzy candidates are generated for games without an exact normalized match.
    All theme groups remain eligible, including groups already used by other games.
    Manual overrides are explicit user decisions and can
    override automatic eligibility; use None to block a game entirely.
    """
    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")
    if not 0 <= review_min_score <= 1 or not 0 <= differing_min_score <= 1:
        raise ValueError("Both similarity thresholds must be in [0, 1].")
    overrides = overrides or {}
    groups, raw_to_key, cleaned_rows = {}, {}, []
    for row in theme_rows:
        raw = row.get("theme_description")
        cleaned = clean_theme_name(raw)
        key = match_key(cleaned)
        raw_to_key[raw] = key
        cleaned_rows.append({**row, "theme_description_clean": cleaned, "theme_match_key": key})
        if not key:
            continue
        group = groups.setdefault(key, {"key": key, "names": set(), "raw": set()})
        group["names"].add(cleaned)
        group["raw"].add(raw)

    for key, group in groups.items():
        group["label"] = min(group["names"], key=lambda x: (len(x), x))
        group["tokens"] = match_tokens(group["label"])

    games = sorted({r["join_game_name"] for r in game_rows if clean_theme_name(r.get("join_game_name"))})
    unknown = set(overrides) - set(games)
    if unknown:
        raise ValueError(f"Unknown join_game_name values in MANUAL_MATCHES: {sorted(unknown)}")

    outputs, candidate_rows = [], []
    for game in games:
        clean_game = clean_theme_name(game)
        target_tokens = match_tokens(clean_game)
        blocked = game in overrides and overrides[game] is None
        selected_key = None
        if game in overrides and not blocked:
            raw_override = overrides[game]
            if not raw_to_key.get(raw_override):
                raise ValueError(f"Invalid raw theme_description override for {game!r}: {raw_override!r}")
            selected_key = raw_to_key[raw_override]

        ranked = []
        if not blocked:
            exact_key = match_key(clean_game)
            candidate_keys = [exact_key] if exact_key in groups else groups.keys()
            for key in candidate_keys:
                comparison = compare_title_tokens(
                    target_tokens, groups[key]["tokens"],
                    review_min_score, differing_min_score,
                )
                if comparison is not None:
                    score, method, differing_score = comparison
                    ranked.append((score, key, method, differing_score))
            ranked.sort(key=lambda x: (-x[0], x[1]))
        exact = [key for score, key, method, diff in ranked if method == "normalized_exact"]

        if blocked:
            status = "blocked"
        elif selected_key is not None:
            status = "manual"
        elif len(exact) == 1:
            selected_key = exact[0]
            status = "normalized_exact"
        elif len(exact) > 1:
            status = "review_ambiguous_exact"
        elif ranked:
            status = "review_fuzzy"
        else:
            status = "unmatched"

        best = ranked[0] if ranked else None
        second = ranked[1] if len(ranked) > 1 else None
        chosen = groups.get(selected_key)
        suggested = groups[best[1]] if best else None
        outputs.append({
            "join_game_name": game, "join_game_name_clean": clean_game,
            "theme_match_status": status, "matched_theme_key": selected_key,
            "matched_theme_description_clean": chosen["label"] if chosen else None,
            "matched_theme_descriptions": sorted(chosen["raw"]) if chosen else [],
            "suggested_theme_description_clean": suggested["label"] if suggested else None,
            "suggested_theme_descriptions": sorted(suggested["raw"]) if suggested else [],
            "suggestion_score": round(best[0], 4) if best else None,
            "suggestion_margin": round(best[0] - second[0], 4) if second else None,
        })
        for rank, (score, key, method, differing_score) in enumerate(ranked[:max(top_n, len(exact))], 1):
            target_only, theme_only = _unshared_words(target_tokens, groups[key]["tokens"])
            candidate_rows.append({
                "join_game_name": game, "candidate_rank": rank,
                "join_game_name_clean": clean_game,
                "candidate_theme_key": key,
                "candidate_theme_description_clean": groups[key]["label"],
                "candidate_theme_descriptions": sorted(groups[key]["raw"]),
                "similarity_score": round(score, 4), "match_method": method,
                "differing_text_similarity": round(differing_score, 4) if differing_score is not None else None,
                "target_unshared_words": target_only,
                "theme_unshared_words": theme_only,
            })
    return cleaned_rows, outputs, candidate_rows


def run_theme_matching(theme_source_df, game_source_df, manual_matches=None,
                       top_n=5, review_min_score=0.80, differing_min_score=0.70):
    """Run in Databricks after loading the two source Spark DataFrames.

    theme_source_df: theme_description_review, filtered_df, or monthly_theme_counts;
                     must contain theme_description.
    game_source_df: like_games_df; must contain join_game_name.
                    to_sales_name_match is preserved but cannot bypass the title comparison.
    manual_matches: {exact_join_game_name: exact_raw_theme_description};
                    use None to explicitly block a game's match.
    top_n: maximum suggestions per game without an exact normalized match.
    review_min_score: full-title character similarity threshold (0 to 1).
    differing_min_score: character similarity required between nonshared words
                         when both titles have nonshared words (0 to 1).
    Fuzzy suggestions are NEVER accepted automatically, even at score 1.0.

    Only DISTINCT names are collected; installation and monthly data stay in Spark.
    Designed for the supplied datasets (744 games and 1,867 descriptions).
    Returns Spark DataFrames; does not write tables or change either input.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, DoubleType, IntegerType, ArrayType,
    )

    if "theme_description" not in theme_source_df.columns:
        raise ValueError("theme_source_df must contain theme_description.")
    if "join_game_name" not in game_source_df.columns:
        raise ValueError("game_source_df must contain join_game_name.")

    spark_session = game_source_df.sparkSession
    themes = theme_source_df.select("theme_description").distinct()
    name_columns = ["join_game_name"]
    if "to_sales_name_match" in game_source_df.columns:
        name_columns.append("to_sales_name_match")
    games = game_source_df.select(*name_columns).distinct()

    theme_rows = [r.asDict() for r in themes.collect()]
    game_rows = [r.asDict() for r in games.collect()]
    clean_rows, summary_rows, candidate_rows = build_theme_mapping(
        theme_rows, game_rows, overrides=manual_matches, top_n=top_n,
        review_min_score=review_min_score, differing_min_score=differing_min_score,
    )

    def string_field(name):
        return StructField(name, StringType(), True)

    def array_field(name):
        return StructField(name, ArrayType(StringType()), True)

    clean_schema = StructType([
        string_field("theme_description"),
        string_field("theme_description_clean"),
        string_field("theme_match_key"),
    ])
    summary_schema = StructType([
        string_field("join_game_name"), string_field("join_game_name_clean"),
        string_field("theme_match_status"), string_field("matched_theme_key"),
        string_field("matched_theme_description_clean"),
        array_field("matched_theme_descriptions"),
        string_field("suggested_theme_description_clean"),
        array_field("suggested_theme_descriptions"),
        StructField("suggestion_score", DoubleType(), True),
        StructField("suggestion_margin", DoubleType(), True),
    ])
    candidate_schema = StructType([
        string_field("join_game_name"),
        string_field("join_game_name_clean"),
        StructField("candidate_rank", IntegerType(), True),
        string_field("candidate_theme_key"),
        string_field("candidate_theme_description_clean"),
        array_field("candidate_theme_descriptions"),
        StructField("similarity_score", DoubleType(), True),
        string_field("match_method"),
        StructField("differing_text_similarity", DoubleType(), True),
        array_field("target_unshared_words"),
        array_field("theme_unshared_words"),
    ])

    theme_lookup_df = spark_session.createDataFrame(clean_rows, clean_schema)
    theme_match_summary_df = spark_session.createDataFrame(summary_rows, summary_schema)
    theme_match_candidates_df = spark_session.createDataFrame(candidate_rows, candidate_schema)

    # One summary row per original game name. Theme keys are NOT unique across games.
    accepted_matches_df = theme_match_summary_df.where(F.col("matched_theme_key").isNotNull())
    game_to_raw_theme_df = accepted_matches_df.select(
        "join_game_name", "matched_theme_key", "matched_theme_description_clean",
        "theme_match_status",
        F.explode("matched_theme_descriptions").alias("theme_description"),
    )

    # A rerun replaces only columns created by this matching routine.
    summary_columns = [c for c in theme_match_summary_df.columns if c != "join_game_name"]
    games_with_matches_df = (
        game_source_df.drop(*summary_columns)
        .join(theme_match_summary_df, "join_game_name", "left")
    )
    themes_with_clean_names_df = (
        theme_source_df.drop("theme_description_clean", "theme_match_key")
        .join(theme_lookup_df, "theme_description", "left")
    )

    return {
        "theme_lookup_df": theme_lookup_df,
        "themes_with_clean_names_df": themes_with_clean_names_df,
        "theme_match_summary_df": theme_match_summary_df,
        "theme_match_candidates_df": theme_match_candidates_df,
        "partial_theme_match_candidates_df": theme_match_candidates_df.where(
            F.col("match_method") == "partial_fuzzy_review"
        ),
        "accepted_matches_df": accepted_matches_df,
        "game_to_raw_theme_df": game_to_raw_theme_df,
        "like_games_with_premium_df": games_with_matches_df,
    }


# EXAMPLE: Run in another Databricks cell after the definitions above.
# Both input DataFrames must be loaded in that notebook.
#
# MANUAL_MATCHES = {
#     # "join_game_name from your data": "exact raw theme_description from your data",
#     # "a game you want left unmatched": None,
# }
# results = run_theme_matching(
#     theme_source_df=theme_description_review,
#     game_source_df=like_games_df,
#     manual_matches=MANUAL_MATCHES,
#     top_n=5,
#     review_min_score=0.80,
#     differing_min_score=0.70,
# )
# display(results["theme_match_summary_df"])
# display(results["theme_match_candidates_df"].orderBy("join_game_name", "candidate_rank"))
# display(results["partial_theme_match_candidates_df"].orderBy("join_game_name", "candidate_rank"))
# like_games_with_premium_df = results["like_games_with_premium_df"]
#
# For monthly premium rows, join by the ORIGINAL description:
# premium_monthly_by_game_df = monthly_theme_counts.join(
#     results["game_to_raw_theme_df"], "theme_description", "inner"
# )
# This intentionally lets different join_game_name values reuse the same theme's rows.


In [0]:
results = run_theme_matching(
    theme_source_df=theme_description_review,
    game_source_df=like_games_df,
    top_n=5,
    review_min_score=0.80,       # Full-title similarity
    differing_min_score=0.70,    # Similarity between differing parts
)

display(
    results["partial_theme_match_candidates_df"]
    .orderBy("join_game_name", "candidate_rank")
)

like_games_with_premium_df = results["like_games_with_premium_df"]

In [0]:
# The lookup has one row per original description, so this left join preserves
# all filtered_df rows and does not multiply lease-unit values. Unmatched names
# remain null and are retained as a separate group in the monthly output.
_filtered_without_eilers_df = filtered_df.drop(
    "theme_name_eilers", "theme_name_eilers_all"
)
filtered_df = (
    _filtered_without_eilers_df
    .join(F.broadcast(theme_name_eilers_lookup_df), "theme_description", "left")
    .withColumn(
        "theme_name_eilers_all",
        F.coalesce(F.col("theme_name_eilers_all"), F.array().cast("array<string>")),
    )
    .select(
        *_filtered_without_eilers_df.columns,
        "theme_name_eilers",
        "theme_name_eilers_all",
    )
)

# Collapse the weekly/source rows to one record per month and Eilers game name.
# Sum numeric measures from the source table, excluding numeric identifiers and
# calendar labels that should not be added together.
non_measure_numeric_columns = {
    "id", "year", "month_number", "week_number", "week_of_year",
    "fiscal_year", "fiscal_month", "fiscal_week",
}
measure_columns = [
    field.name
    for field in filtered_df.schema.fields
    if field.name not in non_measure_numeric_columns
    and not field.name.lower().endswith("_id")
    and field.name != date_col
    and field.name.lower() not in {"month", "yearmonth", "week", "date"}
    and field.dataType.typeName() in {
        "byte", "short", "integer", "long", "float", "double", "decimal"
    }
]

monthly_by_eilers_theme_df = (
    filtered_df
    .groupBy("month", "theme_name_eilers")
    .agg(
        *[
            F.sum(F.col(column_name)).alias(column_name)
            for column_name in measure_columns
        ],
        F.count(F.lit(1)).alias("source_row_count"),
        F.countDistinct("theme_description").alias("source_theme_description_count"),
        F.sort_array(F.collect_set("theme_description")).alias("source_theme_descriptions"),
    )
)

# Save one monthly row per Eilers theme. Each rerun replaces the destination table.
(
    monthly_by_eilers_theme_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(path_final)
)

print(f"Saved monthly Eilers-theme aggregation to {path_final}")
display(monthly_by_eilers_theme_df.orderBy("month", "theme_name_eilers"))
display(theme_name_eilers_lookup_df.orderBy("theme_description"))


In [0]:
display(monthly_by_eilers_theme_df.filter(F.col("theme_name_eilers")=='coyote moon classic hits'))